# Deeper investigation — Reshaping product tags

**Learner exercise** · [All exercises](../../index.html) · [Setup](../../README.md)

Optional. Complete [Exercise 2](../02-clean-keys.ipynb) and its **Save and finish** cell first. This investigation uses the same saved work; it does not replace your core pipeline.

Complete **Your code**, run the **Check** cells, and open hints when needed. Replace `todo(...)` with your answer. Do not use **Run All** while tasks remain unfinished.

Run the supplied setup first. End with **Save and finish**; the next notebook loads your saved functions, so this kernel can be closed.

## Setup — supplied

Select the lab's `.venv` kernel. Stop Spark in the previous notebook before closing it. This uses the `create_spark` helper explained in [Exercise 0](../00-spark-session.ipynb). Missing earlier work? Use an explicit [catch-up step](../../RECOVERY.md).

In [ ]:
from pathlib import Path
import sys

# Support opening the complete repository or its labs folder in VS Code.
LAB_ROOT = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent, parent / 'labs')
     if (candidate / 'workshop_runtime.py').is_file()),
    None,
)
if LAB_ROOT is None:
    raise FileNotFoundError('Open the complete labs project in VS Code; a notebook alone is not enough.')
if str(LAB_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB_ROOT))

from uuid import uuid4

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

import lab_checks as check
from arrival_files import publish_arrival
from lab_checks import todo
from lab_workspace import Workspace
from workshop_runtime import DATA_ROOT, create_spark, finish_query, new_run, spark_path

workspace = Workspace(solutions=False)
RUN_ROOT = new_run()
spark = create_spark(RUN_ROOT)
raw = spark.read.parquet(spark_path(DATA_ROOT / "sales.parquet"))
raw_products = spark.read.parquet(spark_path(DATA_ROOT / "products.parquet"))
print(f"Spark {spark.version}; inputs: {DATA_ROOT.name}; notebook ready")

---
<a id="extension-tags"></a>
## Your task

Read the supplied product-tags CSV. Split `tags_raw` on the literal `|`, explode it into one `tag` per row, and keep `product_id` and `tag` as `product_tags`. Create `distinct_tags` too.

Expected: six product/tag associations and five distinct tags. Explain what one output row means. Do not join this result into the sales report and sum amounts without deciding how multi-tag sales should be attributed.

In [ ]:
tag_input = (
    spark.read.option("header", True)
    .schema("product_id STRING, tags_raw STRING")
    .csv(spark_path(DATA_ROOT / "extras/product_tags.csv"))
)
tag_input.show(truncate=False)

In [ ]:
product_tags = todo("Split the tags, explode the array, and select product_id plus tag")
distinct_tags = todo("Select distinct tag values")

In [ ]:
check.tags(product_tags, distinct_tags)

<details><summary>Hint</summary>

`split` accepts a regular expression: escape the pipe so it means a literal separator. `explode` changes row grain from product to product/tag association.

</details>

<a id="finish"></a>
## Save and finish

Run once the core checks pass, whether or not you did the optional section. This saves your functions or stream handoff, then stops this notebook’s queries and Spark. Your work remains in `learner_work/`.

In [ ]:
for active_query in spark.streams.active:
    active_query.stop()
spark.stop()
print("Session stopped; exercise files are under", RUN_ROOT.relative_to(LAB_ROOT))

Return to [all exercises](../../index.html).

After your attempt, compare the separate [worked solution](../../solutions/deeper/product-tags.ipynb).